In [1]:
# Knowledge Graph Similarity Analysis
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import json
from itertools import combinations
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import networkx as nx

In [2]:
# Configuration
CLOBBER = True  # Set to True to overwrite existing results
DATA_DIR = Path("../../data_products/knowledge_triples_sustcrit.csv")  # Directory with your knowledge graph files
RESULTS_FILE = "../../data_products/kg_similarities.pkl"
FEATURES_FILE = "../../data_products/kg_features.pkl"

In [3]:
# Use sentence-transformers/all-mpnet-base-v2 for triple embeddings
from sentence_transformers import SentenceTransformer

try:
    print("Loading sentence-transformers/all-mpnet-base-v2...")
    model = SentenceTransformer('all-mpnet-base-v2')
    print("✓ Successfully loaded all-mpnet-base-v2")
    available_model = "all-mpnet-base-v2"
except Exception as e:
    print(f"✗ Failed to load all-mpnet-base-v2: {e}")
    # Fallback to smaller model
    try:
        print("Trying fallback: all-MiniLM-L6-v2...")
        model = SentenceTransformer('all-MiniLM-L6-v2')
        print("✓ Successfully loaded all-MiniLM-L6-v2")
        available_model = "all-MiniLM-L6-v2"
    except Exception as e2:
        print(f"✗ Failed to load fallback: {e2}")
        available_model = None

print(f"\nUsing model: {available_model}")

Loading sentence-transformers/all-mpnet-base-v2...
✓ Successfully loaded all-mpnet-base-v2

Using model: all-mpnet-base-v2


In [4]:
trips_df = pd.read_csv(DATA_DIR)
grouped = trips_df.groupby('file')
# Create a list of DataFrames from the grouped object
list_of_dfs = [group_df for group_name, group_df in grouped]

In [5]:
G_list = []
for file in list_of_dfs:
    G = nx.DiGraph()
    zob = zip(file.subject, file.predicate, file.object)
    for s, p, o in zob:
       G.add_edge(s, o, label=p)
    G_list.append(G)


In [6]:
def extract_graph_features(triples_df):
    """Extract simple structural features from a knowledge graph"""
    features = {}
    
    # Basic counts
    features['num_triples'] = len(triples_df)
    features['num_entities'] = len(set(triples_df['subject'].tolist() + triples_df['object'].tolist()))
    features['num_relations'] = len(triples_df['predicate'].unique())
    
    # Relation distribution
    rel_counts = triples_df['predicate'].value_counts()
    features['relation_entropy'] = -sum((rel_counts/len(triples_df)) * np.log2(rel_counts/len(triples_df)))
    features['most_common_relation_ratio'] = rel_counts.iloc[0] / len(triples_df) if len(rel_counts) > 0 else 0
    
    # Create networkx graph for structural features
    G = nx.DiGraph()
    for _, row in triples_df.iterrows():
        G.add_edge(row['subject'], row['object'], label=row['predicate'])
    
    if len(G.nodes()) > 0:
        features['avg_degree'] = sum(dict(G.degree()).values()) / len(G.nodes())
        features['num_components'] = nx.number_weakly_connected_components(G)
        
        # Convert to undirected for clustering coefficient
        G_undirected = G.to_undirected()
        features['clustering_coeff'] = nx.average_clustering(G_undirected)
    else:
        features['avg_degree'] = 0
        features['num_components'] = 0
        features['clustering_coeff'] = 0
    
    return features

def jaccard_similarity(set1, set2):
    """Compute Jaccard similarity between two sets"""
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0

# Main processing
print(f"CLOBBER mode: {CLOBBER}")

CLOBBER mode: True


In [7]:
# Use the sentence transformer model for triple-level embeddings with graph context
print(f"Using {available_model} for contextual triple embeddings...")

def get_all_neighbors_context(triple, graph, max_context=None):
    """Get context triples from all neighbors of the triple's entities"""
    subject, predicate, obj = triple
    context_triples = []
    
    # Get all triples involving the subject
    if subject in graph.nodes():
        for neighbor in graph.neighbors(subject):
            edge_data = graph.get_edge_data(subject, neighbor)
            if edge_data and 'label' in edge_data:
                context_triples.append((subject, edge_data['label'], neighbor))
        
        # Also get incoming edges to subject
        for predecessor in graph.predecessors(subject):
            edge_data = graph.get_edge_data(predecessor, subject)
            if edge_data and 'label' in edge_data:
                context_triples.append((predecessor, edge_data['label'], subject))
    
    # Get all triples involving the object
    if obj in graph.nodes():
        for neighbor in graph.neighbors(obj):
            edge_data = graph.get_edge_data(obj, neighbor)
            if edge_data and 'label' in edge_data:
                context_triples.append((obj, edge_data['label'], neighbor))
        
        # Also get incoming edges to object
        for predecessor in graph.predecessors(obj):
            edge_data = graph.get_edge_data(predecessor, obj)
            if edge_data and 'label' in edge_data:
                context_triples.append((predecessor, edge_data['label'], obj))
    
    # Remove the original triple from context and duplicates
    original_triple = (subject, predicate, obj)
    context_triples = [t for t in context_triples if t != original_triple]
    context_triples = list(set(context_triples))  # Remove duplicates
    
    # Return all neighbors (no max limit)
    return context_triples

def precompute_all_triple_embeddings(all_triples_dfs):
    """Pre-compute embeddings for all unique triples across all graphs"""
    print("Pre-computing embeddings for all unique triples...")
    
    # Collect all unique triples
    all_unique_triples = set()
    for triples_df in all_triples_dfs:
        for _, row in triples_df.iterrows():
            triple = (row['subject'], row['predicate'], row['object'])
            all_unique_triples.add(triple)
    
    print(f"Found {len(all_unique_triples)} unique triples to embed")
    
    # Convert to list and create sentences
    unique_triples_list = list(all_unique_triples)
    triple_sentences = [f"{s} {p} {o}" for s, p, o in unique_triples_list]
    
    # Batch encode all unique triples
    print("Encoding all unique triples in batch...")
    embeddings = model.encode(triple_sentences, show_progress_bar=True)
    
    # Create lookup dictionary
    embedding_cache = {}
    for triple, embedding in zip(unique_triples_list, embeddings):
        embedding_cache[triple] = embedding
    
    print(f"Cached {len(embedding_cache)} triple embeddings")
    return embedding_cache

def encode_triples_with_cached_embeddings(triples_df, graph, embedding_cache, primary_weight=0.6, context_weight=0.4):
    """Encode all triples with their summed graph context using cached embeddings"""
    contextual_embeddings = []
    
    for _, row in triples_df.iterrows():
        triple = (row['subject'], row['predicate'], row['object'])
        
        # Get cached primary embedding
        primary_embedding = embedding_cache[triple]
        
        # Get all context neighbors
        context_triples = get_all_neighbors_context(triple, graph)
        
        if context_triples:
            # Get cached context embeddings and sum them
            context_embeddings = [embedding_cache[ctx_triple] for ctx_triple in context_triples 
                                 if ctx_triple in embedding_cache]
            
            if context_embeddings:
                summed_context_embedding = np.sum(context_embeddings, axis=0)
                # Weighted combination: 60% primary + 40% summed context
                final_embedding = (primary_weight * primary_embedding + 
                                  context_weight * summed_context_embedding)
            else:
                # No valid context embeddings found
                final_embedding = primary_embedding
        else:
            # No context available - use only primary triple
            final_embedding = primary_embedding
        
        contextual_embeddings.append(final_embedding)
    
    return np.array(contextual_embeddings)

def compute_contextual_triple_similarity_cached(triples1, graph1, triples2, graph2, embedding_cache):
    """Compute similarity between two sets of triples using cached contextual embeddings"""
    # Encode all triples from both graphs with their summed contexts using cache
    embeddings1 = encode_triples_with_cached_embeddings(triples1, graph1, embedding_cache)
    embeddings2 = encode_triples_with_cached_embeddings(triples2, graph2, embedding_cache)
    
    if len(embeddings1) == 0 or len(embeddings2) == 0:
        return 0.0
    
    # Compute pairwise similarities
    similarity_matrix = cosine_similarity(embeddings1, embeddings2)
    
    # Use maximum similarity for each triple (best match approach)
    max_similarities = similarity_matrix.max(axis=1)
    
    # Return average of best matches
    return max_similarities.mean()

# Use G_list object instead of reading files
kg_graphs = G_list
graph_dataframes = list_of_dfs

# Check if we should skip processing
if not CLOBBER and Path(RESULTS_FILE).exists():
    print(f"Results file {RESULTS_FILE} already exists and CLOBBER=False. Loading existing results...")
    with open(RESULTS_FILE, 'rb') as f:
        all_similarities = pickle.load(f)
    with open(FEATURES_FILE, 'rb') as f:
        all_features = pickle.load(f)
else:
    print(f"Processing {len(kg_graphs)} knowledge graphs with cached embeddings...")
    
    # Pre-compute all embeddings once
    embedding_cache = precompute_all_triple_embeddings(graph_dataframes)
    
    # Extract features for each graph
    all_features = {}
    all_triples = {}
    
    for i, (graph, triples_df) in enumerate(zip(kg_graphs, graph_dataframes)):
        print(f"Processing graph {i+1}/{len(kg_graphs)}")
        
        # Get the file name from the dataframe (assuming first row has the file name)
        if 'file' in triples_df.columns:
            graph_id = triples_df['file'].iloc[0]
        else:
            graph_id = f"graph_{i}"
            
        features = extract_graph_features(triples_df)
        
        all_features[graph_id] = features
        all_triples[graph_id] = triples_df
    
    # Compute pairwise similarities
    all_similarities = {}
    graph_ids = list(all_features.keys())
    
    total_pairs = len(list(combinations(graph_ids, 2)))
    print(f"Computing cached contextual similarities for {total_pairs} graph pairs...")
    
    for i, (id1, id2) in enumerate(combinations(graph_ids, 2)):
        if i % 10 == 0:  # More frequent updates since this will be slower
            print(f"  Progress: {i}/{total_pairs}")
            
        similarities = {}
        
        # Find corresponding graphs
        graph1 = kg_graphs[graph_ids.index(id1)]
        graph2 = kg_graphs[graph_ids.index(id2)]
        
        # Compute contextual triple-level semantic similarity using cached embeddings
        contextual_sim = compute_contextual_triple_similarity_cached(
            all_triples[id1], graph1, all_triples[id2], graph2, embedding_cache
        )
        similarities['cached_contextual_semantic'] = contextual_sim
        
        # Keep some of the original measures for comparison
        similarities['triple_jaccard'] = jaccard_similarity(
            set(all_triples[id1].apply(lambda x: (x['subject'], x['predicate'], x['object']), axis=1)),
            set(all_triples[id2].apply(lambda x: (x['subject'], x['predicate'], x['object']), axis=1))
        )
        
        # Size similarity
        size1 = len(all_triples[id1])
        size2 = len(all_triples[id2])
        similarities['size_similarity'] = 1 - abs(size1 - size2) / max(size1, size2)
        
        # Add structural feature similarity
        feat1 = np.array(list(all_features[id1].values()))
        feat2 = np.array(list(all_features[id2].values()))
        structural_sim = cosine_similarity([feat1], [feat2])[0][0]
        similarities['structural_similarity'] = structural_sim
        
        all_similarities[(id1, id2)] = similarities
    
    # Save results
    with open(RESULTS_FILE, 'wb') as f:
        pickle.dump(all_similarities, f)
    with open(FEATURES_FILE, 'wb') as f:
        pickle.dump(all_features, f)
    
    print(f"Results saved to {RESULTS_FILE} and {FEATURES_FILE}")

# Analysis and visualization
print("\nSummary Statistics:")
similarity_df = pd.DataFrame(all_similarities).T
print(similarity_df.describe())

print("\nMost similar graph pairs (by cached contextual semantic similarity):")
top_similar = similarity_df.nlargest(10, 'cached_contextual_semantic')
print(top_similar[['triple_jaccard', 'cached_contextual_semantic', 'structural_similarity', 'size_similarity']])

print("\nGraph features summary:")
features_df = pd.DataFrame(all_features).T
print(features_df.describe())

print("\nDone! Check 'similarity_df['cached_contextual_semantic']' for cached contextual similarity results.")

Using all-mpnet-base-v2 for contextual triple embeddings...
Processing 117 knowledge graphs with cached embeddings...
Pre-computing embeddings for all unique triples...
Found 50602 unique triples to embed
Encoding all unique triples in batch...


Batches:   0%|          | 0/1582 [00:00<?, ?it/s]

Cached 50602 triple embeddings
Processing graph 1/117
Processing graph 2/117
Processing graph 3/117
Processing graph 4/117
Processing graph 5/117
Processing graph 6/117
Processing graph 7/117
Processing graph 8/117
Processing graph 9/117
Processing graph 10/117
Processing graph 11/117
Processing graph 12/117
Processing graph 13/117
Processing graph 14/117
Processing graph 15/117
Processing graph 16/117
Processing graph 17/117
Processing graph 18/117
Processing graph 19/117
Processing graph 20/117
Processing graph 21/117
Processing graph 22/117
Processing graph 23/117
Processing graph 24/117
Processing graph 25/117
Processing graph 26/117
Processing graph 27/117
Processing graph 28/117
Processing graph 29/117
Processing graph 30/117
Processing graph 31/117
Processing graph 32/117
Processing graph 33/117
Processing graph 34/117
Processing graph 35/117
Processing graph 36/117
Processing graph 37/117
Processing graph 38/117
Processing graph 39/117
Processing graph 40/117
Processing graph 4

In [15]:
similarity_df

cached_contextual_semantic  \
unfiltered_dependency_edgelist_0007.RDS unfiltered_dependency_edgelist_0008.RDS                    0.696057   
                                        unfiltered_dependency_edgelist_0009.RDS                    0.660036   
                                        unfiltered_dependency_edgelist_0011.RDS                    0.683912   
                                        unfiltered_dependency_edgelist_0012.RDS                    0.664728   
                                        unfiltered_dependency_edgelist_0013.RDS                    0.657537   
...                                                                                                     ...   
unfiltered_dependency_edgelist_0149.RDS unfiltered_dependency_edgelist_0155.RDS                    0.652596   
                                        unfiltered_dependency_edgelist_0156.RDS                    0.723732   
unfiltered_dependency_edgelist_0150.RDS unfiltered_dependency_edgelist_0155.RDS                    0.582399   
                                        unfiltered_dependency_edgelist_0156.RDS                    0.601902   
unfiltered_dependency_edgelist_0155.RDS unfiltered_dependency_edgelist_0156.RDS                    0.670454   

                                                                                 triple_jaccard  \
unfiltered_dependency_edgelist_0007.RDS unfiltered_dependency_edgelist_0008.RDS        0.021004   
                                        unfiltered_dependency_edgelist_0009.RDS        0.001314   
                                        unfiltered_dependency_edgelist_0011.RDS        0.001917   
                                        unfiltered_dependency_edgelist_0012.RDS        0.006494   
                                        unfiltered_dependency_edgelist_0013.RDS        0.004071   
...                                                                                         ...   
unfiltered_dependency_edgelist_0149.RDS unfiltered_dependency_edgelist_0155.RDS        0.000000   
                                        unfiltered_dependency_edgelist_0156.RDS        0.000000   
unfiltered_dependency_edgelist_0150.RDS unfiltered_dependency_edgelist_0155.RDS        0.000000   
                                        unfiltered_dependency_edgelist_0156.RDS        0.003717   
unfiltered_dependency_edgelist_0155.RDS unfiltered_dependency_edgelist_0156.RDS        0.000000   

                                                                                 size_similarity  \
unfiltered_dependency_edgelist_0007.RDS unfiltered_dependency_edgelist_0008.RDS         0.914347   
                                        unfiltered_dependency_edgelist_0009.RDS         0.668094   
                                        unfiltered_dependency_edgelist_0011.RDS         0.399487   
                                        unfiltered_dependency_edgelist_0012.RDS         0.749465   
                                        unfiltered_dependency_edgelist_0013.RDS         0.620985   
...                                                                                          ...   
unfiltered_dependency_edgelist_0149.RDS unfiltered_dependency_edgelist_0155.RDS         0.723926   
                                        unfiltered_dependency_edgelist_0156.RDS         0.401361   
unfiltered_dependency_edgelist_0150.RDS unfiltered_dependency_edgelist_0155.RDS         0.610487   
                                        unfiltered_dependency_edgelist_0156.RDS         0.908163   
unfiltered_dependency_edgelist_0155.RDS unfiltered_dependency_edgelist_0156.RDS         0.554422   

                                                                                 structural_similarity  
unfiltered_dependency_edgelist_0007.RDS unfiltered_dependency_edgelist_0008.RDS               0.998966  
                                        unfiltered_dependency_edgelist_0009.RDS               0.997638  
                                        unf

In [19]:
similarity_df.to_csv('../../data_products/triple_similarity.csv', index=True) 

In [21]:
features_df.to_csv('../../data_products/triple_features.csv', index=True) 